Configure the local data snapshots as described in the README before running this notebook.

# Risk regime detection and forecasting in the Brazilian market

Do regime probabilities improve risk forecasts and net strategy performance for BOVA11? Run the notebook with the local research snapshot configured in the README.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable,"-m","pip","-q","install","exchange-calendars","hmmlearn","arch"])

In [ ]:
from local_snapshots import load_local_snapshot

SNAPSHOT = load_local_snapshot('Ibovespa_Risk_Regimes.ipynb', 'SNAPSHOT')


In [ ]:
import io,base64,zlib,shutil,json,hashlib,inspect
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.special import logsumexp
from scipy.stats import norm
from hmmlearn.hmm import GaussianHMM
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from arch.bootstrap import CircularBlockBootstrap

ASSET_REGIME = "BOVA11"
THRESHOLD = pd.Timestamp("2025-01-01")
SEED = 42
VOL_TARGET = .12
QUARANTINE_BOVA11 = {'2026-10-26': {'reason': 'OHLC differs across providers. The official opening price is unconfirmed.', 'open_yahoo': 140.0, 'open_alternativa': 169.0, 'maximum_alternativa': 176.6, 'minimum_alternativa': 165.06, 'close_alternativo': 172.42, 'source_alternativa': 'https://www.cotab3.com.br/cotacoes?months=12&ticker=BOVA11', 'reference_oficial': 'https://www.b3.com.br/pt_br/market-data-e-indexes/servicos-de-data/market-data/historical/mercado-a-vista/cotacoes-historicas/'}}
COST_PER_SIDE = .0015
HIGH_VOLATILITY_MULTIPLIER = .60
OUTPUT_DIR = Path("results_regimes")/"P1"/ASSET_REGIME
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def read_snapshot(name):
    content = zlib.decompress(base64.b64decode(SNAPSHOT[name]))
    df = pd.read_csv(io.BytesIO(content),index_col=0,parse_dates=True)
    if name == "signal":
        df = df.rename(columns={"execucao": "execution", "previsao_bruta": "raw_forecast", "sinal": "signal"})
        df.index.name = "origin"
    df.index = df.index.astype("datetime64[ns]")
    assert df.index.is_monotonic_increasing and df.index.is_unique
    return df

bova,ibov = read_snapshot("BOVA11"),read_snapshot("IBOV")
try:
    from market_calendar import calendar_and_sessions
except ModuleNotFoundError as error:
    if error.name != "market_calendar":
        raise
    import exchange_calendars as xcals

    # Fallback generated from market_calendar.py.
    def calendar_and_sessions(name, start, end):
        """Return the exchange calendar and session labels between two inclusive dates."""
        start, end = pd.Timestamp(start), pd.Timestamp(end)
        if any(pd.isna(day) or day.tz is not None or day != day.normalize()
               for day in (start, end)) or start > end:
            raise ValueError("Enter local dates without times in ascending order.")
        margin = pd.Timedelta(days=15)
        calendar = xcals.get_calendar(name, start=start - margin, end=end + margin)
        sessions = calendar.sessions_in_range(start, end)
        if sessions.tz is not None:
            sessions = sessions.tz_localize(None)
        return calendar, sessions

cal, sessions = calendar_and_sessions("BVMF", ibov.index.min(), ibov.index.max())
assert sessions.equals(ibov.index) and bova.index.isin(sessions).all()
assert ASSET_REGIME in {"BOVA11","IBOV"}
price = (bova["Adj Close"] if ASSET_REGIME=="BOVA11" else ibov.Close).reindex(sessions)
assert np.isfinite(price.dropna()).all() and price.dropna().gt(0).all()
r = np.log(price).diff()
pre = r.loc[r.index<THRESHOLD]
train = max([s.dropna() for _,s in pre.groupby(pre.isna().cumsum())],key=len)*100
assert len(train)>=252 and train.index.max()<THRESHOLD
print("Contiguous training observations:",len(train),"| end:",train.index.max().date(),
      "| sessions without a closing price:",int(price.isna().sum()))

## 1. Regime detection and next-session forecasts

The Gaussian HMM estimates state-specific means and variances. Markov switching uses a common mean and state-dependent variance. States are ordered by fitted variance.

In [ ]:
hmm = GaussianHMM(n_components=3,covariance_type="diag",n_iter=400,random_state=SEED)
hmm.fit(train.to_numpy()[:,None])
ms = MarkovRegression(train.to_numpy(),k_regimes=3,trend="c",
                      switching_trend=False,switching_variance=True)
# Recent statsmodels versions accept a dedicated random generator.
# Older versions use deterministic default initialization and log the difference.
if "rng" in inspect.signature(ms.fit).parameters:
    modo_ms = "busca_com_rng_explicito"
    ms_fit = ms.fit(search_reps=8,rng=np.random.default_rng(SEED),
                    em_iter=8,maxiter=400,disp=False)
else:
    modo_ms = "standard_start_without_random_search"
    ms_fit = ms.fit(search_reps=0,em_iter=8,maxiter=400,disp=False)
ms_params = dict(zip(ms.param_names,ms_fit.params))
assert ms_fit.mle_retvals["converged"], "Markov switching did not converge."
print("HMM stopping criterion:",hmm.monitor_.converged,"| Markov switching converged:")

try:
    from regime_filter import filter_regimes
except ModuleNotFoundError as error:
    if error.name != "regime_filter":
        raise
    # Fallback generated from regime_filter.py.
    def filter_regimes(series, mean, var, A, initial):
        """Sort states by variance and return current and next-session probabilities."""
        observations = np.asarray(series, dtype=float)
        mean, var = np.asarray(mean, dtype=float), np.asarray(var, dtype=float)
        A, initial = np.asarray(A, dtype=float), np.asarray(initial, dtype=float)
        k = var.size
        if (observations.ndim != 1 or np.isinf(observations).any() or not k
            or mean.shape != (k,) or var.shape != (k,)
            or A.shape != (k, k) or initial.shape != (k,)):
            raise ValueError("Invalid dimensions or observations for the regime filter.")
        if (not np.isfinite(mean).all() or not np.isfinite(var).all() or (var <= 0).any()
            or not np.isfinite(A).all() or (A < 0).any() or not np.allclose(A.sum(axis=1), 1)
            or not np.isfinite(initial).all() or (initial < 0).any() or not np.isclose(initial.sum(), 1)):
            raise ValueError("Invalid Gaussian distributions or transition probabilities.")

        order = np.argsort(var)
        mean, var = mean[order], var[order]
        A = A[np.ix_(order, order)]
        prior = initial[order].copy()
        hoje = np.empty((len(observations), k))
        amanha = np.empty_like(hoje)
        dispersion = np.sqrt(var)
        for i, value in enumerate(observations):
            if pd.notna(value):
                logpeso = np.log(np.maximum(prior, 1e-300)) + norm.logpdf(value, mean, dispersion)
                prior = np.exp(logpeso - logsumexp(logpeso))
            hoje[i] = prior
            prior = prior @ A
            amanha[i] = prior
        assert np.allclose(amanha.sum(axis=1), 1)
        return hoje, amanha, mean, var, A

ms_A = ms.regime_transition_matrix(ms_fit.params)[:,:,0].T
parameters = {
    "HMM":(hmm.means_.ravel(),hmm.covars_.reshape(3,-1)[:,0],hmm.transmat_,hmm.startprob_),
    "Markov Switching":(np.repeat(ms_params["const"],3),
        np.array([ms_params[f"sigma2[{i}]"] for i in range(3)]),ms_A,ms.initial_probabilities(ms_fit.params))
}
names = ["low","Mean","high"]
probabilities = {}
def hash_objeto(objeto):
    return hashlib.sha256(json.dumps(objeto,sort_keys=True,allow_nan=False).encode()).hexdigest()

manifest = {
    "revisao":"P1.1",
    "asset_regime":ASSET_REGIME,
    "sha256_snapshot":hashlib.sha256(zlib.decompress(base64.b64decode(SNAPSHOT[ASSET_REGIME]))).hexdigest(),
    "start_fit":str(train.index.min().date()),
    "end_fit":str(train.index.max().date()),
    "n_fit":len(train),
    "first_target_test":str(THRESHOLD.date()),
    "seed":SEED,
    "versoes":{name:version(name) for name in
               ["numpy","pandas","scipy","statsmodels","hmmlearn","exchange-calendars","arch"]},
    "models":{},
}
# Export the latest forecast origin before its target return is observed.
targets_exportacao = pd.Series(sessions,index=sessions).shift(-1)
targets_exportacao.iloc[-1] = cal.next_session(sessions[-1])
outside_fit = targets_exportacao.ge(THRESHOLD)&(sessions>=train.index.max())
metadata = pd.DataFrame({
    "target_date":targets_exportacao,
    "sample_status":np.where(outside_fit,"out_of_sample","in_sample_fit"),
    "target_observed":r.shift(-1).notna(),
    "return_value_origin_observed":r.notna(),
    "end_fit":train.index.max(),
},index=sessions)
variances = pd.DataFrame({"Historical 21":r.pow(2).rolling(21).mean(),
    "EWMA":r.pow(2).groupby(r.isna().cumsum()).transform(
        lambda x:x.ewm(alpha=.06,min_periods=21,adjust=False).mean()).where(r.notna())})
for name,param in parameters.items():
    current,future,means,vars_,A = filter_regimes(r*100,*param)
    # Causality check: removing later observations leaves the path through the cutoff unchanged.
    threshold = len(r)//2
    _,curta,*_ = filter_regimes(r.iloc[:threshold]*100,*param)
    assert np.allclose(future[:threshold],curta)
    print(name)
    display(pd.DataFrame(A,index=names,columns=names).round(3))
    display(pd.Series(np.sqrt(vars_*252)/100,index=names,name="annualized volatility by state"))
    probabilities[name] = pd.DataFrame(future,index=sessions,columns=names)
    variances[name] = future@(vars_+means**2)/1e4
    details = {
        "means_percentuais":means.tolist(),
        "percentage_variances":vars_.tolist(),
        "transition":A.tolist(),
        "raw_parameters":ms_params if name=="Markov Switching" else {
            "initial":hmm.startprob_.tolist(),"means":hmm.means_.ravel().tolist(),
            "variances":hmm.covars_.reshape(3,-1)[:,0].tolist(),
            "transition":hmm.transmat_.tolist(),
        },
        "log_verossimilhanca":float(ms_fit.llf if name=="Markov Switching"
                                    else hmm.score(train.to_numpy()[:,None])),
        "convergence":bool(ms_fit.mle_retvals["converged"] if name=="Markov Switching"
                            else hmm.monitor_.converged),
        "inicializacao":modo_ms if name=="Markov Switching" else "hmm_random_state",
    }
    identity = hash_objeto({"model":name,"details":details,
        "snapshot":manifest["sha256_snapshot"],"start_fit":manifest["start_fit"],
        "end_fit":manifest["end_fit"],"versoes":manifest["versoes"]})
    manifest["models"][name] = details|{"id_fit":identity}
    filtered = pd.DataFrame(current,index=sessions,columns=names).add_prefix("filtered_D_")
    export = metadata.join(filtered).join(probabilities[name].add_prefix("predicted_D1_"))
    export["id_fit"] = identity
    export.to_csv(OUTPUT_DIR/f"probabilities_{name.replace(' ','_')}.csv",index_label="origin")
(OUTPUT_DIR/"manifest_regimes.json").write_text(
    json.dumps(manifest,indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
fig,axs = plt.subplots(2,1,figsize=(12,6),sharex=True)
for ax,(name,probs) in zip(axs,probabilities.items()):
    probs.loc[THRESHOLD:].plot(ax=ax,title=f"{name} · next-session regime probability",ylim=(0,1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"probabilities_D1.png",dpi=140)
plt.show()

## 2. H1 · Forecast accuracy

Because regimes are latent, the predicted second moment is evaluated against an observable daily proxy. Lower QLIKE is better. The paired bootstrap uses blocks of ten exchange sessions.

In [ ]:
target_date = pd.Series(sessions,index=sessions).shift(-1)
observed = r.shift(-1).pow(2)
common = variances.join(observed.rename("observed")).loc[target_date.ge(THRESHOLD)].dropna()
assert len(common)>40
rows = []
for name in variances:
    h = common[name].clip(lower=1e-10)
    base = common.EWMA.clip(lower=1e-10)
    loss = np.log(h)+common.observed/h
    dif = loss-(np.log(base)+common.observed/base)
    ic = CircularBlockBootstrap(10,dif.to_numpy(),seed=SEED).conf_int(
        np.mean,reps=2000,method="percentile").ravel()
    rows.append({"model":name,"n":len(common),"QLIKE":loss.mean(),
        "MAE_variance":(h-common.observed).abs().mean(),"dif_vs_EWMA":dif.mean(),
        "IC_inf":ic[0],"IC_sup":ic[1]})
ranking = pd.DataFrame(rows).sort_values("QLIKE")
display(ranking)
ranking.to_csv(OUTPUT_DIR/"forecast_risk.csv",index=False)

## 3. H2 and H3 · Position sizing in BOVA11

Signals are determined at the close. Positions enter at the next open and exit at that session's close. Regime sizing uses a weight of 1 minus 0.60 times the probability of high volatility.

In [ ]:
bars = bova.reindex(sessions)
ret_oc = (bars.Close/bars.Open-1).shift(-1)
high = probabilities["HMM"].high
weight_regime = 1-HIGH_VOLATILITY_MULTIPLIER*high
weights = pd.DataFrame({"Fixed 100%":1.,"Regime-based sizing":weight_regime},index=sessions)
for name in ["Historical 21","EWMA","HMM"]:
    weights[f"Vol target · {name}"] = (VOL_TARGET/np.sqrt(252*variances[name].clip(lower=1e-10))).clip(upper=1)
eligible = target_date.ge(THRESHOLD)&~target_date.isin(pd.to_datetime(list(QUARANTINE_BOVA11)))
data = weights.join(ret_oc.rename("ret")).loc[eligible].dropna()
assert len(data)>40

try:
    from intraday_backtest import empirical_cvar, equity_drawdown, simulate_intraday
except ModuleNotFoundError as error:
    if error.name != "intraday_backtest":
        raise
    # Fallback generated from intraday_backtest.py.
    def equity_drawdown(equity):
        """Measure drawdown from the peak, including initial equity of one."""
        return equity / equity.cummax().clip(lower=1) - 1


    def empirical_cvar(returns, tail=0.05):
        """Average the worst losses with fractional weight at the tail boundary."""
        values = np.asarray(returns, dtype=float)
        if values.ndim != 1 or not values.size or not np.isfinite(values).all():
            raise ValueError("CVaR requires a nonempty vector of finite returns.")
        if not 0 < tail < 1:
            raise ValueError("The tail probability must be between zero and one.")
        losses = np.sort(-values)[::-1]
        mass = len(losses) * tail
        integers = int(mass)
        return (losses[:integers].sum() + (mass - integers) * losses[integers]) / mass


    def simulate_intraday(weights, returns, cost_per_side, *, include_details=False):
        """Use aligned exposures and returns, including sessions held in cash."""
        if (weights.empty or not weights.index.equals(returns.index)
            or not weights.index.is_unique or not weights.index.is_monotonic_increasing
            or not weights.columns.is_unique):
            raise ValueError("Weights and returns must share a chronological panel with unique dates.")
        if not np.isfinite(cost_per_side) or cost_per_side < 0:
            raise ValueError("Cost per side must be finite and nonnegative.")
        if not np.isfinite(weights.to_numpy(dtype=float)).all() or not np.isfinite(returns).all():
            raise ValueError("Weights and returns must be finite for every session.")

        raw = weights.mul(returns, axis=0)
        costs = 2 * cost_per_side * weights.abs()
        net = raw - costs
        equity = (1 + net).cumprod()
        deviation = net.std()
        exposure = weights.abs().mean()
        metrics = pd.DataFrame({
            "net_return": equity.iloc[-1] - 1,
            "vol_annual": deviation * np.sqrt(252),
            "sharpe": np.sqrt(252) * net.mean() / deviation.replace(0, np.nan),
            "drawdown": equity_drawdown(equity).min(),
            "VaR_95": -net.quantile(0.05),
            "CVaR_95": net.apply(empirical_cvar),
            "exposure_mean": exposure,
            "mean_turnover": 2 * exposure,
            "trades": weights.ne(0).sum(),
        })
        if include_details:
            metrics["gross_return"] = (1 + raw).prod() - 1
            metrics["total_daily_costs"] = costs.sum()
            metrics["session_count"] = len(weights)
        return metrics, equity, net

w = data.drop(columns="ret").copy()
for name in ["Regime-based sizing","Vol target · HMM"]:
    w[f"Exposure-matched fixed sizing · {name}"] = w[name].mean()
table_risk,curve_risk,_ = simulate_intraday(w,data.ret,COST_PER_SIDE)
display(table_risk)
table_risk.to_csv(OUTPUT_DIR/"risk_managed_trading.csv")

record = read_snapshot("signal")
idx = record.index.intersection(data.index)
assert (pd.to_datetime(record.loc[idx,"execution"])==target_date.loc[idx]).all()
prev = record.loc[idx,"raw_forecast"]
side = np.sign(prev).where(prev.abs()>.0035,0)
assert side.ne(0).eq(record.loc[idx,"signal"]).all()
weighted = side*weight_regime.loc[idx]
constant = weighted.abs().loc[side.ne(0)].mean() if side.ne(0).any() else 0.
signals = pd.DataFrame({"Frozen signal":side,"signal + regime":weighted,
                      "Signal with matched fixed sizing":side*constant},index=idx)
table_alpha,curve_alpha,_ = simulate_intraday(signals,data.ret.loc[idx],COST_PER_SIDE)
display(table_alpha)
table_alpha.to_csv(OUTPUT_DIR/"trading_signal_frozen.csv")
pd.concat({"weights":signals,"return_value":data.ret.loc[idx].to_frame()},axis=1).to_csv(
    OUTPUT_DIR/"signals_and_exposure.csv",index_label="origin")
fig,ax = plt.subplots(2,1,figsize=(12,7))
curve_risk[["Fixed 100%","Regime-based sizing","Vol target · EWMA","Vol target · HMM"]].plot(
    ax=ax[0],title="BOVA11 intraday risk management after costs")
curve_alpha.plot(ax=ax[1],title="Frozen XGBoost with risk-based position sizing")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"trading_comparison.png",dpi=140)
plt.show()

## 4. P1.2 · Regime stability before 2025

Compare two- and three-state HMM and Markov switching fits over two chronological windows of 63 targets each. Every fit uses only earlier returns.

In [ ]:
from itertools import combinations
import warnings

SEEDS_STABILITY = [42,137,251]
SIZE_VALIDATION = 63
targets_validation = train.index[-2*SIZE_VALIDATION:]
assert len(targets_validation)==2*SIZE_VALIDATION and targets_validation.max()<THRESHOLD
windows_validation = [targets_validation[:SIZE_VALIDATION],targets_validation[SIZE_VALIDATION:]]
suporta_rng = "rng" in inspect.signature(ms.fit).parameters

def fit_para_stability(familia,k,seed,historical):
    values = historical.to_numpy()
    if familia == "HMM":
        fit = GaussianHMM(n_components=k,covariance_type="diag",n_iter=400,
                             random_state=seed).fit(values[:,None])
        param = (fit.means_.ravel(),fit.covars_.reshape(k,-1)[:,0],
                 fit.transmat_,fit.startprob_)
        converged = bool(fit.monitor_.converged and fit.monitor_.iter<fit.n_iter)
        return param,converged,fit.score(values[:,None])
    model = MarkovRegression(values,k_regimes=k,trend="c",
                             switching_trend=False,switching_variance=True)
    kwargs = {"rng":np.random.default_rng(seed),"search_reps":8} if suporta_rng else {}
    fit = model.fit(em_iter=8,maxiter=400,disp=False,**kwargs)
    coef = dict(zip(model.param_names,fit.params))
    param = (np.repeat(coef["const"],k),np.array([coef[f"sigma2[{i}]"] for i in range(k)]),
             model.regime_transition_matrix(fit.params)[:,:,0].T,
             model.initial_probabilities(fit.params))
    return param,bool(fit.mle_retvals["converged"]),fit.llf

fits,states_stability,comparisons = [],[],[]
prob_por_seed = {}
for window,targets in enumerate(windows_validation,start=1):
    origins = sessions[sessions.get_indexer(targets)-1]
    assert target_date.loc[origins].tolist()==targets.tolist()
    hist = train.loc[train.index<targets.min()]
    assert hist.index.max()<=origins.min()
    series = (r.loc[hist.index.min():targets.max()]*100)
    pos = series.index.get_indexer(origins)
    pos_target = series.index.get_indexer(targets)
    assert (pos>=0).all() and (pos_target>=0).all() and (pos_target==pos+1).all()
    observed_pct = series.loc[targets].to_numpy()
    assert np.isfinite(observed_pct).all()
    for familia in ["HMM","Markov Switching"]:
        seeds = SEEDS_STABILITY if familia=="HMM" or suporta_rng else [SEED]
        for k in [2,3]:
            group = (window,familia,k)
            prob_por_seed[group] = {}
            for seed in seeds:
                row = {"window":window,"model":familia,"states":k,"seed":seed,
                         "n_fit":len(hist),"end_fit":hist.index.max().date(),
                         "start_target":targets.min().date(),"end_target":targets.max().date(),
                         "n_validation":len(targets)}
                try:
                    with warnings.catch_warnings(record=True) as avisos:
                        warnings.simplefilter("always")
                        param,converged,llf = fit_para_stability(familia,k,seed,hist)
                        current,future,means,var,A = filter_regimes(series,*param)
                    p = future[pos]
                    q = current[pos_target]
                    h = np.maximum(p@(var+means**2)/1e4,1e-10)
                    losses = np.log(h)+(observed_pct/100)**2/h
                    logp = logsumexp(np.log(np.maximum(p,1e-300))+
                                      norm.logpdf(observed_pct[:,None],means,np.sqrt(var)),axis=1)
                    if not np.isfinite(np.r_[losses,logp,llf]).all():
                        raise ValueError("Nonfinite metrics.")
                    row.update({"converged":converged,"loglik_fit":llf,
                                  "QLIKE":losses.mean(),"neg_log_score_pct":-logp.mean(),
                                  "alertas_fit":"; ".join(sorted({str(a.message) for a in avisos}))})
                    if converged:
                        prob_por_seed[group][seed] = p[:,-1]
                    for i in range(k):
                        diagonal = float(A[i,i])
                        occupancy = float(q[:,i].mean())
                        dominancia = float((q.argmax(axis=1)==i).mean())
                        ratio = float(var[i]/var[i-1]) if i else np.nan
                        states_stability.append({**{key:row[key] for key in
                            ["window","model","states","seed"]},"order_variance":i+1,
                            "converged":converged,"vol_annual":np.sqrt(var[i]*252)/100,
                            "occupancy_filtered":occupancy,"frequency_dominante":dominancia,
                            "duration_teorica":1/(1-diagonal) if diagonal<1 else np.inf,
                            "ratio_variance_previous":ratio,
                            "alerta_occupancy":occupancy<.05 or dominancia==0,
                            "alerta_separacao":bool(i and ratio<1.20)})
                except (ValueError,RuntimeError,AssertionError,np.linalg.LinAlgError) as error:
                    row.update({"converged":False,"alertas_fit":str(error)})
                fits.append(row)
for (window,familia,k),por_seed in prob_por_seed.items():
    for (sa,pa),(sb,pb) in combinations(por_seed.items(),2):
        comparisons.append({"window":window,"model":familia,"states":k,
                            "seed_a":sa,"seed_b":sb,
                            "dif_abs_mean_p_high":np.abs(pa-pb).mean()})

fits = pd.DataFrame(fits)
states_stability = pd.DataFrame(states_stability)
divergencias = pd.DataFrame(comparisons,columns=[
    "window","model","states","seed_a","seed_b","dif_abs_mean_p_high"])
fits.to_csv(OUTPUT_DIR/"stability_fits.csv",index=False)
states_stability.to_csv(OUTPUT_DIR/"stability_states.csv",index=False)
divergencias.to_csv(OUTPUT_DIR/"stability_entre_seeds.csv",index=False)
keys = ["window","model","states"]
summary_stability = fits.groupby(keys).converged.agg(
    tentativas="size",convergencias="sum").reset_index()
valid = fits.loc[fits.converged]
if not valid.empty:
    performance = valid.groupby(keys).agg(
        median_QLIKE=("QLIKE","median"),QLIKE_min=("QLIKE","min"),QLIKE_max=("QLIKE","max"),
        median_negative_log_score=("neg_log_score_pct","median")).reset_index()
    summary_stability = summary_stability.merge(performance,on=keys,how="left")
if not divergencias.empty:
    divergence = divergencias.groupby(keys).dif_abs_mean_p_high.max().rename(
        "longest_dif_mean_p_high").reset_index()
    summary_stability = summary_stability.merge(divergence,on=keys,how="left")
display(summary_stability)
summary_stability.to_csv(OUTPUT_DIR/"stability_summary.csv",index=False)

## 5. P1.3 · Walk-forward refits and observable risk

Compare a two-state Markov switching model fitted before 2025 with a version refitted every 63 origins when enough contiguous observations are available.

In [ ]:
STEP_REFIT = 63
MIN_RETURNS_REFIT = 126
THRESHOLD_QUANTIL = .90

origins_evaluate = sessions[target_date.ge(THRESHOLD).to_numpy()]
assert len(origins_evaluate)>=STEP_REFIT
threshold_extremo = float(train.abs().quantile(THRESHOLD_QUANTIL))  # percentual
frequency_fit = float(train.abs().gt(threshold_extremo).mean())
param_fixed,converged,llf_fixed = fit_para_stability("Markov Switching",2,SEED,train)
if not converged:
    raise RuntimeError("The fixed two-state Markov switching model did not converge.")

def forecast_risk_em_extremo(param,origins):
    # The path through the last origin uses only observed returns.
    series = r.loc[:origins[-1]]*100
    _,future,means,vars_,_ = filter_regimes(series,*param)
    pos = series.index.get_indexer(origins)
    assert (pos>=0).all()
    p = future[pos]
    dispersion = np.sqrt(vars_)
    event_por_state = norm.cdf(-threshold_extremo,means,dispersion)+norm.sf(threshold_extremo,means,dispersion)
    return pd.DataFrame({"variance":p@(vars_+means**2)/1e4,
                         "prob_extremo":p@event_por_state},index=origins)

fixed = forecast_risk_em_extremo(param_fixed,origins_evaluate)
param_current = param_fixed
end_current = train.index.max()
partes_refit,log_refit = [],[]
for i in range(0,len(origins_evaluate),STEP_REFIT):
    block = origins_evaluate[i:i+STEP_REFIT]
    available = r.loc[:block[0]]
    last_block = available.groupby(available.isna().cumsum()).get_group(
        available.isna().sum()).dropna()*100
    status = "insufficient_contiguous_history"
    details = ""
    llf = np.nan
    if len(last_block)>=MIN_RETURNS_REFIT:
        if last_block.index.max()<=end_current:
            status = "current_fit_already_uses_history"
        else:
            try:
                novo,ok,llf = fit_para_stability("Markov Switching",2,SEED,last_block)
                # Validate parameters and filter before accepting the new fit.
                if ok:
                    forecast_risk_em_extremo(novo,block[:1])
                    param_current,end_current = novo,last_block.index.max()
                    status = "refitted"
                else:
                    status = "failed_to_converge"
            except (ValueError,RuntimeError,AssertionError,np.linalg.LinAlgError) as error:
                status,details = "falha_no_fit",str(error)
    assert end_current<=block[0]
    p = forecast_risk_em_extremo(param_current,block)
    p["end_fit"] = end_current
    partes_refit.append(p)
    log_refit.append({"first_origin":block[0],"last_origin":block[-1],
        "n_contiguous_available":len(last_block),"end_fit_usado":end_current,
        "status":status,"loglik_tentativa":llf,"details":details})
refit = pd.concat(partes_refit)
pd.DataFrame(log_refit).to_csv(OUTPUT_DIR/"walk_forward_refits.csv",index=False)

ewma = variances.EWMA.reindex(origins_evaluate)
prob_ewma = 2*norm.sf(threshold_extremo/100/np.sqrt(ewma.clip(lower=1e-10)))
return_value_target = r.shift(-1).reindex(origins_evaluate)
frame = pd.DataFrame({"target_date":target_date.loc[origins_evaluate],
    "return_value_target":return_value_target,"variance_observed":return_value_target.pow(2),
    "variance_fixed":fixed.variance,"variance_refit":refit.variance,
    "variance_ewma":ewma,"prob_fixed":fixed.prob_extremo,"prob_refit":refit.prob_extremo,
    "prob_ewma":prob_ewma,"prob_frequency":frequency_fit,
    "end_fit_fixed":train.index.max(),"end_fit_refit":refit.end_fit})
frame["event"] = return_value_target.abs().gt(threshold_extremo/100).where(return_value_target.notna())
frame.to_csv(OUTPUT_DIR/"walk_forward_forecasts.csv",index_label="origin")
sample = frame.dropna()
assert len(sample)>100
assert (sample.end_fit_refit<=sample.index).all()

# Fallback generated from risk_statistics.py for standalone use in Colab.
try:
    from risk_statistics import session_adjacency, paired_mean_interval
except ModuleNotFoundError as error:
    if error.name!="risk_statistics":
        raise
    def session_adjacency(sessions, dates):
        """Mark adjacent exchange sessions while preserving gaps."""
        positions = sessions.get_indexer(dates)
        steps = np.diff(positions)
        if len(positions)==0 or (positions<0).any() or (steps<=0).any():
            raise ValueError("The sample requires known, unique sessions in chronological order.")
        return steps==1

    def paired_mean_interval(difference,adjacent,block=10,rep=3000,seed=42):
        """Apply a circular bootstrap within each contiguous segment, preserving its size."""
        difference = np.asarray(difference,dtype=float)
        adjacent = np.asarray(adjacent,dtype=bool)
        if (difference.ndim!=1 or not len(difference) or not np.isfinite(difference).all()
            or adjacent.shape!=(len(difference)-1,) or block<1 or rep<1):
            raise ValueError("Invalid paired sample, adjacency, or configuration.")
        rng = np.random.default_rng(seed)
        thresholds = np.flatnonzero(~adjacent)+1
        sums = np.zeros(rep)
        for segment in np.split(difference,thresholds):
            k = min(block,len(segment))
            starts = rng.integers(len(segment),size=(rep,int(np.ceil(len(segment)/k))))
            indexes = ((starts[:,:,None]+np.arange(k))%len(segment)).reshape(rep,-1)
            sums += segment[indexes[:,:len(segment)]].sum(axis=1)
        return np.quantile(sums/len(difference),[.025,.975])

def interval_matched(difference,block=10,n_boot=3000):
    adjacent = session_adjacency(sessions,difference.index)
    return paired_mean_interval(difference,adjacent,block,n_boot,SEED)

def qlike(h):
    h = h.clip(lower=1e-10)
    return np.log(h)+sample.variance_observed/h

base_risk = qlike(sample.variance_ewma)
rows_risk = []
for name in ["fixed","refit","ewma"]:
    loss = qlike(sample[f"variance_{name}"])
    dif = loss-base_risk
    lo,hi = interval_matched(dif)
    rows_risk.append({"model":name,"n":len(loss),"QLIKE":loss.mean(),
        "dif_vs_ewma":dif.mean(),"IC_inf":lo,"IC_sup":hi})
table_walk = pd.DataFrame(rows_risk).sort_values("QLIKE")
display(table_walk)
table_walk.to_csv(OUTPUT_DIR/"walk_forward_qlike.csv",index=False)

y_event = sample.event.astype(int)
rows_event,bins = [],[]
for name in ["fixed","refit","ewma","frequency"]:
    p = sample[f"prob_{name}"].clip(1e-10,1-1e-10)
    brier = (p-y_event)**2
    loss_log = -(y_event*np.log(p)+(1-y_event)*np.log1p(-p))
    dif_brier = brier-(sample.prob_frequency-y_event)**2
    lo,hi = interval_matched(dif_brier)
    rows_event.append({"model":name,"n":len(y_event),"events":int(y_event.sum()),
        "Brier":brier.mean(),"log_loss":loss_log.mean(),"prob_mean":p.mean(),
        "frequency_observed":y_event.mean(),"dif_brier_vs_frequency":dif_brier.mean(),
        "IC_inf":lo,"IC_sup":hi})
    groups = pd.DataFrame({"p":p,"event":y_event}).groupby(
        pd.cut(p,[0,.05,.10,.20,.40,1.],include_lowest=True),observed=True)
    bin = groups.agg(n=("event","size"),prob_mean=("p","mean"),
                      frequency_observed=("event","mean")).reset_index(names="bin")
    bins.append(bin.assign(model=name))
calibration = pd.DataFrame(rows_event).sort_values("Brier")
display(calibration)
calibration.to_csv(OUTPUT_DIR/"calibration_event_extremo.csv",index=False)
pd.concat(bins,ignore_index=True).to_csv(OUTPUT_DIR/"calibration_bins_event_extremo.csv",index=False)
(OUTPUT_DIR/"config_walk_forward.json").write_text(json.dumps({
    "revisao":"P1.3","step_refit":STEP_REFIT,"min_returns_refit":MIN_RETURNS_REFIT,
    "event_quantile_train":THRESHOLD_QUANTIL,"threshold_absoluto_logret":threshold_extremo/100,
    "frequency_train":frequency_fit,"end_train_fixed":str(train.index.max().date()),
    "seed":SEED,"bootstrap_block":10},indent=2),encoding="utf-8")
print(f"extreme event: |log return| > {threshold_extremo/100:.2%} | {int(y_event.sum())}/{len(y_event)} in the common sample")

## 6. P1.7 · Execution and risk benchmarks

All intraday policies receive the same BOVA11 open-to-close returns, with exposure capped at 100% and a 15-basis-point cost per side. The passive benchmark uses its own holding period and cost convention.

In [ ]:
WINDOW_WEIGHT_FIXED = 126
# Exclude the same unreliable session from OHLC-based execution.
oc = (bars.Close/bars.Open-1).where(~bars.index.isin(pd.to_datetime(list(QUARANTINE_BOVA11))))
var_oc = oc.pow(2).groupby(oc.isna().cumsum()).transform(
    lambda s:s.ewm(alpha=.06,adjust=False,min_periods=21).mean()).where(oc.notna())
variances_h2 = pd.DataFrame({"Close-to-close EWMA":variances.EWMA,"Intraday EWMA":var_oc})
weights_h2 = (VOL_TARGET/np.sqrt(252*variances_h2.clip(lower=1e-10))).clip(upper=1)
target_oc = oc.shift(-1)
common_h2 = weights_h2.join(target_oc.rename("ret")).dropna()
pre_h2 = common_h2.loc[target_date.loc[common_h2.index].lt(THRESHOLD)].tail(WINDOW_WEIGHT_FIXED)
assert len(pre_h2)==WINDOW_WEIGHT_FIXED
sample_h2 = common_h2.loc[target_date.loc[common_h2.index].ge(THRESHOLD)]
assert len(sample_h2)>100
assert target_date.loc[pre_h2.index].max()<target_date.loc[sample_h2.index].min()
w_h2 = sample_h2.drop(columns="ret").copy()
w_h2["Fixed 100%"] = 1.
for name in variances_h2:
    w_h2[f"Fixed pretest sizing · {name}"] = pre_h2[name].mean()

h2,curves_h2,net_returns_h2 = simulate_intraday(w_h2,sample_h2.ret,COST_PER_SIDE,include_details=True)
h2["weight_calibrated_pre_test"] = [pre_h2[c.split(" · ")[1]].mean() if c.startswith("Fixed pretest sizing") else np.nan
                                  for c in h2.index]
display(h2)
h2.to_csv(OUTPUT_DIR/"H2_intraday_benchmarks.csv",index_label="policy")
curves_h2.assign(target_date=target_date.loc[curves_h2.index]).to_csv(OUTPUT_DIR/"H2_curves_intraday.csv",index_label="origin")
comparison_h2 = []
for name,estimate in variances_h2.loc[sample_h2.index].items():
    h = estimate.clip(lower=1e-10)
    q = (np.log(h)+sample_h2.ret.pow(2)/h).mean()
    difference = net_returns_h2[name]-net_returns_h2[f"Fixed pretest sizing · {name}"]
    lo,hi = interval_matched(difference)
    comparison_h2.append({"model":name,"QLIKE_oc":q,"n":len(sample_h2),
        "daily_net_gain_vs_fixed_pretest":difference.mean(),"IC_inf":lo,"IC_sup":hi})
comparison_h2 = pd.DataFrame(comparison_h2)
display(comparison_h2)
comparison_h2.to_csv(OUTPUT_DIR/"H2_forecast_e_controls.csv",index=False)

passive_price = bova["Adj Close"].reindex(sessions[sessions>=THRESHOLD])
assert len(passive_price.dropna())>=2
entry,output = passive_price.first_valid_index(),passive_price.last_valid_index()
passive_price = passive_price.loc[entry:output]
passive_raw = passive_price/passive_price.loc[entry]
passive_net = passive_raw*(1-COST_PER_SIDE)
passive_net.loc[output] *= 1-COST_PER_SIDE
ret_day = passive_net.pct_change(fill_method=None)
ret_day.loc[entry] = -COST_PER_SIDE
observed_passive = ret_day.dropna().to_numpy()
cvar = empirical_cvar(observed_passive)
passive = pd.DataFrame([{"entry_close":entry,"output_close":output,
    "n_sessions":len(passive_price),"returns_daily_observed":len(observed_passive),
    "gross_return":passive_raw.loc[output]-1,"net_return":passive_net.loc[output]-1,
    "impacto_costs_no_return_value":passive_raw.loc[output]-passive_net.loc[output],
    "drawdown":equity_drawdown(passive_net).min(),
    "CVaR_95":cvar,"vol_annual":ret_day.std()*np.sqrt(252),
    "sharpe":np.sqrt(252)*ret_day.mean()/ret_day.std() if ret_day.std()>0 else np.nan,
    "exposure_mean":1.,"mean_turnover":2/len(passive_price),"trades":1}])
display(passive)
passive.to_csv(OUTPUT_DIR/"H2_benchmark_passive.csv",index=False)
pd.DataFrame({"equity_raw":passive_raw,"equity_net":passive_net}).to_csv(
    OUTPUT_DIR/"H2_curve_passiva.csv",index_label="data")

# Preserva todas as oportunidades e o lado do sinal frozen.
ret_h3 = data.ret.loc[signals.index]
h3,curves_h3,net_returns_h3 = simulate_intraday(signals,ret_h3,COST_PER_SIDE,include_details=True)
h3.to_csv(OUTPUT_DIR/"H3_execution_frozen.csv",index_label="policy")
assets_h3 = signals["Frozen signal"].ne(0)
if assets_h3.any():
    traded = signals.loc[assets_h3]
    x = traded["signal + regime"].abs().to_numpy()
    sides = traded["Frozen signal"].to_numpy()
    r_h3 = ret_h3.loc[assets_h3].to_numpy()
    rng = np.random.default_rng(SEED)
    permuted = np.tile(x,(5000,1))
    for side_control in [-1,1]:
        mascara = sides==side_control
        permuted[:,mascara] = [rng.permutation(x[mascara]) for _ in range(len(permuted))]
    control = np.prod(1+permuted*(sides*r_h3-2*COST_PER_SIDE),axis=1)-1
    q = np.quantile(control,[.025,.5,.975])
    difference = net_returns_h3["signal + regime"]-net_returns_h3["Signal with matched fixed sizing"]
    lo,hi = interval_matched(difference)
    observed_h3 = h3.loc["signal + regime","net_return"]
    table_h3 = pd.DataFrame([{"trades":int(assets_h3.sum()),"return_value_regime":observed_h3,
        "control_q025":q[0],"control_median":q[1],"control_q975":q[2],
        "percentile_descritivo":(control<=observed_h3).mean(),
        "difference_daily_vs_fixed_matched":difference.mean(),"IC_inf":lo,"IC_sup":hi}])
    display(table_h3)
    table_h3.to_csv(OUTPUT_DIR/"H3_control_por_side.csv",index=False)
fig,axs = plt.subplots(2,1,figsize=(12,7))
curves_h2[["Close-to-close EWMA","Intraday EWMA","Fixed 100%"]].set_axis(
    pd.DatetimeIndex(target_date.loc[curves_h2.index])).plot(ax=axs[0],title="Daily trading after costs")
passive_net.plot(ax=axs[1],title="Passive BOVA11 equity after costs")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"H2_benchmarks.png",dpi=140)
plt.show()
(OUTPUT_DIR/"config_H2_H3.json").write_text(json.dumps({
    "revisao":"P1.7","window_weight_fixed":WINDOW_WEIGHT_FIXED,"end_calibration_weight":str(pre_h2.index.max().date()),
    "vol_target":VOL_TARGET,"cost_per_side":COST_PER_SIDE,"lambda_ewma":.94,"seed":SEED,
    "permutacoes_por_side":5000},indent=2),encoding="utf-8")
print("Download results:",shutil.make_archive(str(OUTPUT_DIR),"zip",OUTPUT_DIR))
